## Import libraries

In [12]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

## Load cleaned data

In [13]:
df = pd.read_csv(
    "../data/processed/cleaned_orders.csv"
)

df.head()

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,...,total_payment_value,customer_city,customer_state,customer_zip_code_prefix,review_score,has_review,review_comment_title,review_comment_message,review_creation_date,target
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,...,38.71,sao paulo,SP,3149,4.0,1,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,...,141.46,barreiras,BA,47813,4.0,1,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,...,179.12,vianopolis,GO,75265,5.0,1,NaN,NaN,2018-08-18 00:00:00,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,...,72.20,sao goncalo do amarante,RN,59296,5.0,1,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,...,28.62,santo andre,SP,9195,5.0,1,NaN,NaN,2018-02-17 00:00:00,1.0


## Check Dataset Overview

In [16]:
print("Dataset shape:", df.shape)

Dataset shape: (96478, 38)


In [ ]:
print("\nColumns:")

df.columns.tolist()


Columns:


['order_id',
 'customer_id',
 'customer_unique_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delay_days',
 'approval_time_hours',
 'purchase_year',
 'purchase_month',
 'purchase_dayofweek',
 'purchase_hour',
 'order_item_count',
 'product_category',
 'avg_product_weight_g',
 'avg_product_length_cm',
 'avg_product_width_cm',
 'avg_product_height_cm',
 'avg_product_photos_qty',
 'avg_product_description_len',
 'total_item_price',
 'total_freight_value',
 'payment_type',
 'payment_installments',
 'total_payment_value',
 'customer_city',
 'customer_state',
 'customer_zip_code_prefix',
 'review_score',
 'has_review',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'target']

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       96478 non-null  object 
 1   customer_id                    96478 non-null  object 
 2   customer_unique_id             96478 non-null  object 
 3   order_status                   96478 non-null  object 
 4   order_purchase_timestamp       96478 non-null  object 
 5   order_approved_at              96478 non-null  object 
 6   order_delivered_carrier_date   96476 non-null  object 
 7   order_delivered_customer_date  96470 non-null  object 
 8   order_estimated_delivery_date  96478 non-null  object 
 9   delivery_days                  96470 non-null  float64
 10  delay_days                     96470 non-null  float64
 11  approval_time_hours            96478 non-null  float64
 12  purchase_year                  96478 non-null 

## Convert datetime

In [20]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_creation_date'
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors='coerce'
        )

print("Datetime conversion completed!")

Datetime conversion completed!


## Total Order Value

In [21]:
df['total_order_value'] = (
    df['total_item_price']
    + df['total_freight_value']
)

df[['total_order_value']].head()

,total_order_value
0,38.71
1,141.46
2,179.12
3,72.20
4,28.62


## Shipping Ratio

In [22]:
df['shipping_ratio'] = (
    df['total_freight_value']
    / df['total_order_value']
)

df[['shipping_ratio']].head()

,shipping_ratio
0,0.225265
1,0.160894
2,0.107302
3,0.376731
4,0.304682


## Late Delivery Flag

In [23]:
df['is_late_delivery'] = np.where(
    df['delay_days'] > 0,
    1,
    0
)

df[['is_late_delivery']].head()

,is_late_delivery
0,0
1,0
2,0
3,0
4,0


## Installment Flag

In [24]:
df['is_installment'] = np.where(
    df['payment_installments'] > 1,
    1,
    0
)

df[['is_installment']].head()

,is_installment
0,0
1,0
2,1
3,0
4,0


## Heavy Product

In [25]:
df['is_heavy_product'] = np.where(
    df['avg_product_weight_g'] > 1000,
    1,
    0
)

df[['is_heavy_product']].head()

,is_heavy_product
0,0
1,0
2,0
3,0
4,0


## Weekend Purchase

In [26]:
df['is_weekend_purchase'] = np.where(
    df['purchase_dayofweek'] >= 5,
    1,
    0
)

df[['is_weekend_purchase']].head()

,is_weekend_purchase
0,0
1,0
2,0
3,1
4,0


## Expensive Shipping

In [27]:
df['expensive_shipping'] = np.where(
    df['shipping_ratio'] > 0.2,
    1,
    0
)

df[['expensive_shipping']].head()

,expensive_shipping
0,1
1,0
2,0
3,1
4,1


## Product Volume

In [28]:
df['product_volume_cm3'] = (
    df['avg_product_length_cm']
    * df['avg_product_width_cm']
    * df['avg_product_height_cm']
)

df[['product_volume_cm3']].head()

,product_volume_cm3
0,1976.0
1,4693.0
2,9576.0
3,6000.0
4,11475.0


## Product Transparency Score

In [29]:
df['product_info_score'] = (
    df['avg_product_photos_qty']
    + df['avg_product_description_len']
)

df[['product_info_score']].head()

,product_info_score
0,272.0
1,179.0
2,233.0
3,471.0
4,320.0


## Check Missing Values

In [30]:
df.isnull().sum().sort_values(
    ascending=False
).head(20)

review_comment_title             85286
review_comment_message           57576
review_creation_date              8494
review_score                       646
target                             646
delivery_days                        8
delay_days                           8
order_delivered_customer_date        8
order_delivered_carrier_date         2
order_status                         0
order_purchase_timestamp             0
order_approved_at                    0
customer_unique_id                   0
customer_id                          0
order_id                             0
purchase_dayofweek                   0
purchase_month                       0
purchase_year                        0
approval_time_hours                  0
order_estimated_delivery_date        0
dtype: int64

## Remove Unnecessary Columns

In [31]:
df.drop(
    columns=[
        'review_comment_title',
        'review_comment_message'
    ],
    inplace=True,
    errors='ignore'
)

## Remove Missing Target

In [32]:
df.dropna(
    subset=['target'],
    inplace=True
)

## Missing Value Handling

In [33]:
# Fill missing delivered date
if 'order_delivered_customer_date' in df.columns:
    df['order_delivered_customer_date'].fillna(
        df['order_estimated_delivery_date'],
        inplace=True
    )

# Fill missing carrier date
if 'order_delivered_carrier_date' in df.columns:
    df['order_delivered_carrier_date'].fillna(
        df['order_approved_at'],
        inplace=True
    )

# Fill numeric missing
numeric_cols = df.select_dtypes(
    include=np.number
).columns

for col in numeric_cols:
    df[col].fillna(
        df[col].median(),
        inplace=True
    )

In [37]:
df.drop(
    columns=['review_creation_date'],
    inplace=True,
    errors='ignore'
)

Final Missing Check

In [38]:
df.isnull().sum().sort_values(
    ascending=False
).head(20)

order_id                         0
customer_id                      0
customer_unique_id               0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
delivery_days                    0
delay_days                       0
approval_time_hours              0
purchase_year                    0
purchase_month                   0
purchase_dayofweek               0
purchase_hour                    0
order_item_count                 0
product_category                 0
avg_product_weight_g             0
avg_product_length_cm            0
dtype: int64

Save Featured Dataset

In [39]:
df.to_csv(
    "../data/processed/featured_orders.csv",
    index=False
)

print("Feature engineering completed successfully!")

Feature engineering completed successfully!


## Final Check

In [40]:
print(df.shape)

df.head()

(95832, 44)


,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,...,target,total_order_value,shipping_ratio,is_late_delivery,is_installment,is_heavy_product,is_weekend_purchase,expensive_shipping,product_volume_cm3,product_info_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,...,1.0,38.71,0.225265,0,0,0,0,1,1976.0,272.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,...,1.0,141.46,0.160894,0,0,0,0,0,4693.0,179.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,...,1.0,179.12,0.107302,0,1,0,0,0,9576.0,233.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,...,1.0,72.20,0.376731,0,0,0,1,1,6000.0,471.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,...,1.0,28.62,0.304682,0,0,0,0,1,11475.0,320.0
